# Chapter 21 Companion Notebook: Case Studies and Capstone Projects

This notebook reproduces Chapter 21's end-to-end case study on the UCI/Kaggle "Default of Credit Card Clients" dataset (Yeh and Lien, 2009), downloading the real dataset directly from the UCI Machine Learning Repository, including an isotonic recalibration of the plain logistic regression (Section 6b) that fixes its lowest-decile miscalibration without retraining or discarding the model.

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## 1. Download and load the raw dataset

In [1]:
import os
import urllib.request
import numpy as np
import pandas as pd

DATA_URL = "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip"
DATA_DIR = "ch16_data"
ZIP_PATH = os.path.join(DATA_DIR, "uci_credit.zip")
XLS_PATH = os.path.join(DATA_DIR, "default of credit card clients.xls")

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(XLS_PATH):
    import zipfile
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

df = pd.read_excel(XLS_PATH, header=1)
print("raw file has PAY_0:", "PAY_0" in df.columns, " raw file has PAY_1:", "PAY_1" in df.columns)
# The raw file names the most recent repayment-status column PAY_0 and has no PAY_1;
# rename it so the six status columns PAY_1..PAY_6 line up with BILL_AMT1..6 and PAY_AMT1..6.
df = df.rename(columns={"default payment next month": "default", "PAY_0": "PAY_1"})
print("shape:", df.shape)
print("default rate:", df['default'].mean())
df[['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1']].describe().round(1)

raw file has PAY_0: True  raw file has PAY_1: False
shape: (30000, 25)
default rate: 0.2212


,LIMIT_BAL,AGE,BILL_AMT1,PAY_AMT1
count,30000.0,30000.0,30000.0,30000.0
mean,167484.3,35.5,51223.3,5663.6
std,129747.7,9.2,73635.9,16563.3
min,10000.0,21.0,-165580.0,0.0
25%,50000.0,28.0,3558.8,1000.0
50%,140000.0,34.0,22381.5,2100.0
75%,240000.0,41.0,67091.0,5006.0
max,1000000.0,79.0,964511.0,873552.0


## 2. Clean undocumented category codes

In [2]:
print("EDUCATION before cleaning:")
print(df['EDUCATION'].value_counts().sort_index())
print("\nMARRIAGE before cleaning:")
print(df['MARRIAGE'].value_counts().sort_index())

# The data dictionary defines PAY_x = -1 (paid duly) and 1..9 (months overdue) only;
# the columns also contain -2 and 0, which are left as they are (both rank below 1, so neither counts as delinquency).
pay_status = df[[f"PAY_{i}" for i in range(1, 7)]]
print("\nPAY_1..PAY_6 status codes (rows = code, columns = month):")
print(pay_status.apply(lambda s: s.value_counts()).fillna(0).astype(int).sort_index())
print("share of all status entries coded -2 or 0:", round(pay_status.isin([-2, 0]).values.mean(), 4))
print("accounts with at least one -2 or 0 code:", int(pay_status.isin([-2, 0]).any(axis=1).sum()))

df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

print("\nEDUCATION after cleaning:")
print(df['EDUCATION'].value_counts().sort_index())

EDUCATION before cleaning:
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

MARRIAGE before cleaning:
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64

PAY_1..PAY_6 status codes (rows = code, columns = month):
    PAY_1  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6
-2   2759   3782   4085   4348   4546   4895
-1   5686   6050   5938   5687   5539   5740
 0  14737  15730  15764  16455  16947  16286
 1   3688     28      4      2      0      0
 2   2667   3927   3819   3159   2626   2766
 3    322    326    240    180    178    184
 4     76     99     76     69     84     49
 5     26     25     21     35     17     13
 6     11     12     23      5      4     19
 7      9     20     27     58     58     46
 8     19      1      3      2      1      2
share of all status entries coded -2 or 0: 0.6685
accounts with at least one -2 or 0 code: 25939

EDUCATION after cleaning:
EDUCATION
1    10585
2    14030

## 3. Feature engineering

In [3]:
pay_cols = [f"PAY_{i}" for i in [1, 2, 3, 4, 5, 6]]
bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
paid_cols = [f"PAY_AMT{i}" for i in range(1, 7)]

df['max_delinquency'] = df[pay_cols].max(axis=1)
df['avg_utilization'] = (df[bill_cols].div(df['LIMIT_BAL'], axis=0)).mean(axis=1)
df['total_bill'] = df[bill_cols].sum(axis=1)
df['total_paid'] = df[paid_cols].sum(axis=1)
df['pay_to_bill_ratio'] = (df['total_paid'] / df['total_bill'].replace(0, np.nan)).fillna(0).clip(upper=5)

# SEX and MARRIAGE are prohibited bases under ECOA/Regulation B, and AGE is usable there only in narrow,
# specially validated ways, so none of the three is a model input; SEX and AGE are kept aside for the fairness audit.
feature_cols = ['LIMIT_BAL', 'EDUCATION'] + pay_cols + bill_cols + paid_cols + \
    ['max_delinquency', 'avg_utilization', 'total_bill', 'total_paid', 'pay_to_bill_ratio']
print("number of features:", len(feature_cols))

number of features: 25


## 4. Stratified train/test split

In [4]:
from sklearn.model_selection import train_test_split

X = df[feature_cols].copy()
y = df['default'].copy()
sex = df['SEX'].copy()
age = df['AGE'].copy()   # audit attributes only, not model inputs

X_train, X_test, y_train, y_test, sex_train, sex_test, age_train, age_test = train_test_split(
    X, y, sex, age, test_size=0.2, stratify=y, random_state=42)
print("train shape:", X_train.shape, " test shape:", X_test.shape)
print("train default rate:", y_train.mean(), " test default rate:", y_test.mean())

train shape: (24000, 25)  test shape: (6000, 25)
train default rate: 0.22120833333333334  test default rate: 0.22116666666666668


## 5. Model bake-off: logistic regression, random forest, XGBoost

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score, brier_score_loss)
import xgboost as xgb

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {}
lr_plain = LogisticRegression(max_iter=2000).fit(X_train_s, y_train)
models['Logistic (plain)'] = (lr_plain, X_test_s)

lr_bal = LogisticRegression(max_iter=2000, class_weight='balanced').fit(X_train_s, y_train)
models['Logistic (balanced)'] = (lr_bal, X_test_s)

rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1).fit(X_train, y_train)
models['Random Forest'] = (rf, X_test)

xgb_model = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                               colsample_bytree=0.8, eval_metric='logloss', random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
models['XGBoost'] = (xgb_model, X_test)

rows = []
for name, (model, Xte) in models.items():
    proba = model.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred),
        'recall': recall_score(y_test, pred),
        'f1': f1_score(y_test, pred),
        'roc_auc': roc_auc_score(y_test, proba),
        'pr_auc': average_precision_score(y_test, proba),
        'brier': brier_score_loss(y_test, proba),
    })
pd.DataFrame(rows).set_index('model').round(3)

,accuracy,precision,recall,f1,roc_auc,pr_auc,brier
model,,,,,,,
Logistic (plain),0.807,0.683,0.238,0.353,0.720,0.494,0.146
Logistic (balanced),0.731,0.426,0.615,0.503,0.721,0.485,0.205
Random Forest,0.816,0.660,0.350,0.458,0.775,0.554,0.136
XGBoost,0.818,0.665,0.356,0.464,0.778,0.557,0.135


## 6. Calibration: reliability by decile

In [6]:
for name in ['Logistic (plain)', 'XGBoost']:
    model, Xte = models[name]
    proba = model.predict_proba(Xte)[:, 1]
    bins = pd.qcut(proba, 10, duplicates='drop')
    cal = pd.DataFrame({'proba': proba, 'actual': y_test.values, 'bin': bins})
    grouped = cal.groupby('bin', observed=True).agg(mean_pred=('proba', 'mean'), mean_actual=('actual', 'mean'), n=('actual', 'size'))
    print(f"--- {name} ---")
    print(grouped.round(3))
    print()

--- Logistic (plain) ---
                         mean_pred  mean_actual    n
bin                                                 
(-0.0009999809, 0.0563]      0.040        0.135  600
(0.0563, 0.0968]             0.080        0.107  600
(0.0968, 0.129]              0.111        0.168  600
(0.129, 0.156]               0.142        0.087  600
(0.156, 0.177]               0.167        0.118  600
(0.177, 0.192]               0.185        0.120  600
(0.192, 0.246]               0.206        0.148  600
(0.246, 0.331]               0.296        0.265  600
(0.331, 0.47]                0.399        0.393  600
(0.47, 0.992]                0.577        0.670  600

--- XGBoost ---
                                mean_pred  mean_actual    n
bin                                                        
(0.010499999999999999, 0.0488]      0.036        0.045  600
(0.0488, 0.0726]                    0.061        0.075  600
(0.0726, 0.0959]                    0.084        0.083  600
(0.0959, 0.119]       

### 6b. Fixing miscalibration: isotonic recalibration

Recalibrate the plain logistic regression using 5-fold cross-validated isotonic regression (fit within the training set only), and check whether the Brier score and lowest-bin calibration gap improve.

In [7]:
from sklearn.calibration import CalibratedClassifierCV

proba_plain = lr_plain.predict_proba(X_test_s)[:, 1]
brier_plain = brier_score_loss(y_test, proba_plain)

calib_lr = CalibratedClassifierCV(LogisticRegression(max_iter=2000), method='isotonic', cv=5)
calib_lr.fit(X_train_s, y_train)
proba_calib = calib_lr.predict_proba(X_test_s)[:, 1]
brier_calib = brier_score_loss(y_test, proba_calib)

print(f"Plain logistic Brier: {brier_plain:.4f}")
print(f"Isotonic-recalibrated Brier: {brier_calib:.4f}")
print(f"ROC-AUC plain: {roc_auc_score(y_test, proba_plain):.4f}")
print(f"ROC-AUC recalibrated: {roc_auc_score(y_test, proba_calib):.4f}")

bins_calib = pd.qcut(proba_calib, 10, duplicates='drop')
cal_calib = pd.DataFrame({'proba': proba_calib, 'actual': y_test.values, 'bin': bins_calib})
grouped_calib = cal_calib.groupby('bin', observed=True).agg(
    mean_pred=('proba', 'mean'), mean_actual=('actual', 'mean'), n=('actual', 'size'))
print("\nRecalibrated reliability table:")
print(grouped_calib.round(4))

Plain logistic Brier: 0.1460
Isotonic-recalibrated Brier: 0.1419
ROC-AUC plain: 0.7205
ROC-AUC recalibrated: 0.7244

Recalibrated reliability table:
                  mean_pred  mean_actual     n
bin                                           
(0.0128, 0.1045]     0.0983       0.1225   800
(0.1045, 0.1109]     0.1090       0.1205   531
(0.1109, 0.1113]     0.1113       0.1210  1446
(0.1113, 0.1132]     0.1126       0.1089   257
(0.1132, 0.1435]     0.1264       0.1325   566
(0.1435, 0.2523]     0.1800       0.1771   717
(0.2523, 0.2986]     0.2695       0.2525   491
(0.2986, 0.5765]     0.4225       0.3902   592
(0.5765, 0.9206]     0.6865       0.6750   600


## 7. Global explanations: permutation importance and SHAP

In [8]:
from sklearn.inspection import permutation_importance
import shap

perm = permutation_importance(xgb_model, X_test, y_test, n_repeats=10, random_state=42, scoring='roc_auc', n_jobs=-1)
imp_df = pd.DataFrame({'feature': feature_cols, 'importance_mean': perm.importances_mean}).sort_values(
    'importance_mean', ascending=False)
print("Top 10 by permutation importance:")
print(imp_df.head(10).to_string(index=False))

explainer = shap.TreeExplainer(xgb_model)
X_test_sample = X_test.iloc[:1000]
shap_values = explainer(X_test_sample)
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_imp = pd.DataFrame({'feature': feature_cols, 'mean_abs_shap': mean_abs_shap}).sort_values(
    'mean_abs_shap', ascending=False)
print("\nTop 10 by mean |SHAP|:")
print(shap_imp.head(10).to_string(index=False))

Top 10 by permutation importance:
        feature  importance_mean
          PAY_1         0.033747
max_delinquency         0.031750
avg_utilization         0.011762
      BILL_AMT1         0.009448
      LIMIT_BAL         0.005684
       PAY_AMT2         0.004619
     total_bill         0.004442
     total_paid         0.004298
      BILL_AMT5         0.003874
       PAY_AMT3         0.003465

Top 10 by mean |SHAP|:
          feature  mean_abs_shap
  max_delinquency       0.443814
            PAY_1       0.339050
  avg_utilization       0.167244
        BILL_AMT1       0.130010
        LIMIT_BAL       0.118494
         PAY_AMT2       0.101315
         PAY_AMT3       0.082709
         PAY_AMT1       0.073670
       total_paid       0.065664
pay_to_bill_ratio       0.059153


## 8. Individual explanations: a low-risk and a high-risk applicant

In [9]:
idx_low = 0
applicant_low = X_test_sample.iloc[idx_low]
proba_low = xgb_model.predict_proba(X_test_sample.iloc[[idx_low]])[0, 1]
top_low = pd.DataFrame({'feature': feature_cols, 'value': applicant_low.values,
                        'shap': shap_values.values[idx_low]}).sort_values('shap', key=abs, ascending=False)
print("Low-risk applicant, predicted default probability:", round(proba_low, 4))
print(top_low.head(6).to_string(index=False))
print("base value:", explainer.expected_value, " sum shap + base:",
      shap_values.values[idx_low].sum() + explainer.expected_value)

proba_sample = xgb_model.predict_proba(X_test_sample)[:, 1]
idx_high = int(np.argmax(proba_sample))
applicant_high = X_test_sample.iloc[idx_high]
top_high = pd.DataFrame({'feature': feature_cols, 'value': applicant_high.values,
                         'shap': shap_values.values[idx_high]}).sort_values('shap', key=abs, ascending=False)
print("\nHigh-risk applicant, predicted default probability:", round(proba_sample[idx_high], 4))
print(top_high.head(6).to_string(index=False))
print("base value:", explainer.expected_value, " sum shap + base:",
      shap_values.values[idx_high].sum() + explainer.expected_value)

Low-risk applicant, predicted default probability: 0.1307
        feature        value      shap
max_delinquency     0.000000 -0.308778
avg_utilization     0.121113 -0.215403
      LIMIT_BAL 50000.000000  0.174952
          PAY_1    -1.000000 -0.151047
      BILL_AMT1  1540.000000  0.132286
       PAY_AMT1 26060.000000 -0.103337
base value: -1.2719016  sum shap + base: -1.8947152

High-risk applicant, predicted default probability: 0.922
          feature   value     shap
            PAY_1     3.0 1.008909
  max_delinquency     7.0 0.920935
        LIMIT_BAL 10000.0 0.303218
        BILL_AMT1  2300.0 0.233407
            PAY_6     7.0 0.158055
pay_to_bill_ratio     0.0 0.151419
base value: -1.2719016  sum shap + base: 2.4697156


### A worked example: from predicted probability to a provisioning number (EL = PD x LGD x EAD)

In [10]:
LGD = 0.60  # Chapter 9's illustrative loss-given-default assumption, reused here for an unsecured card balance

ead_low = applicant_low["LIMIT_BAL"]
ead_high = applicant_high["LIMIT_BAL"]

el_low = proba_low * LGD * ead_low
el_high = proba_sample[idx_high] * LGD * ead_high

# LIMIT_BAL, and hence EAD and EL, are in New Taiwan dollars (NT$)
print(f"Low-risk applicant:  PD={proba_low:.7f}  EAD=NT${ead_low:,.0f}  LGD={LGD}  ->  EL=NT${el_low:,.2f}")
print(f"High-risk applicant: PD={proba_sample[idx_high]:.7f}  EAD=NT${ead_high:,.0f}  LGD={LGD}  ->  EL=NT${el_high:,.2f}")
print(f"\nEL ratio (high/low): {el_high/el_low:.2f}")
print(f"PD ratio (high/low): {proba_sample[idx_high]/proba_low:.2f}")

Low-risk applicant:  PD=0.1307077  EAD=NT$50,000  LGD=0.6  ->  EL=NT$3,921.23
High-risk applicant: PD=0.9219913  EAD=NT$10,000  LGD=0.6  ->  EL=NT$5,531.95

EL ratio (high/low): 1.41
PD ratio (high/low): 7.05


## 9. Cost-sensitive threshold selection

In [11]:
from sklearn.metrics import confusion_matrix

proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
cost_fn, cost_fp = 500, 50
thresholds = np.arange(0.05, 0.61, 0.01)
cost_table = []
for t in thresholds:
    pred_t = (proba_xgb >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_t).ravel()
    cost_table.append((t, fn, fp, fn * cost_fn + fp * cost_fp))
cost_df = pd.DataFrame(cost_table, columns=['threshold', 'fn', 'fp', 'total_cost'])
best_row = cost_df.loc[cost_df['total_cost'].idxmin()]
show_t = sorted(set([0.05, 0.20, 0.35, 0.50, 0.60, round(best_row['threshold'], 2)]))
print(cost_df[cost_df['threshold'].round(2).isin(show_t)].round(2).to_string(index=False))
print("\ncost-minimizing threshold:", best_row['threshold'], " total cost:", best_row['total_cost'])
print("cost at naive 0.5 threshold:", cost_df.loc[np.isclose(cost_df['threshold'], 0.5), 'total_cost'].values[0])
best_t = best_row['threshold']
t_star = cost_fp / (cost_fp + cost_fn)   # deny when p*cost_fn > (1-p)*cost_fp
pred_star = (proba_xgb >= t_star).astype(int)
tn_s, fp_s, fn_s, tp_s = confusion_matrix(y_test, pred_star).ravel()
print(f"analytic threshold c_FP/(c_FP+c_FN) = {t_star:.4f}, cost there: {fn_s * cost_fn + fp_s * cost_fp}")
print(f"approval rate at the cost-minimizing threshold: {(proba_xgb < best_t).mean():.4f}")
print(f"approval rate at the naive 0.5 threshold: {(proba_xgb < 0.5).mean():.4f}")
nd = (y_test.values == 0)
print(f"share of non-defaulters denied at the cost-minimizing threshold: {(proba_xgb[nd] >= best_t).mean():.4f}")


 threshold  fn   fp  total_cost
      0.05  29 4071      218050
      0.10 129 2905      209750
      0.20 450 1218      285900
      0.35 667  510      359000
      0.50 854  238      438900
      0.60 918  179      467950

cost-minimizing threshold: 0.1  total cost: 209750.0
cost at naive 0.5 threshold: 438900
analytic threshold c_FP/(c_FP+c_FN) = 0.0909, cost there: 214100
approval rate at the cost-minimizing threshold: 0.3162
approval rate at the naive 0.5 threshold: 0.8815
share of non-defaulters denied at the cost-minimizing threshold: 0.6217


## 10. Fairness audit by sex

In [12]:
pred_final = (proba_xgb >= best_t).astype(int)
audit_df = pd.DataFrame({'sex': sex_test.values, 'actual': y_test.values, 'pred_default': pred_final, 'proba': proba_xgb})

results = {}
for s, label in [(1, 'Male'), (2, 'Female')]:
    sub = audit_df[audit_df['sex'] == s]
    approval_rate = (sub['pred_default'] == 0).mean()
    defaulters = sub[sub['actual'] == 1]
    fnr = defaulters['pred_default'].eq(0).mean()
    non_defaulters = sub[sub['actual'] == 0]
    fpr = non_defaulters['pred_default'].eq(1).mean()
    results[label] = approval_rate
    print(f"{label}: n={len(sub)} default_rate={sub['actual'].mean():.4f} approval_rate={approval_rate:.4f} FNR={fnr:.4f} FPR={fpr:.4f}")

air = min(results.values()) / max(results.values())
print("\nadverse impact ratio:", round(air, 4))

Male: n=2402 default_rate=0.2336 approval_rate=0.3085 FNR=0.0891 FPR=0.6247
Female: n=3598 default_rate=0.2129 approval_rate=0.3213 FNR=0.1031 FPR=0.6197

adverse impact ratio: 0.9602


In [13]:
# For contrast only, not a deployable model: the same XGBoost configuration refit WITH SEX, MARRIAGE and AGE
# as inputs, audited at the same threshold, to measure how much leaving them out actually changed.
demo_cols = feature_cols + ['SEX', 'MARRIAGE', 'AGE']
demo_Xtr = df.loc[X_train.index, demo_cols]
demo_Xte = df.loc[X_test.index, demo_cols]
demo_model = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                               colsample_bytree=0.8, eval_metric='logloss', random_state=42, n_jobs=-1)
demo_model.fit(demo_Xtr, y_train)
demo_proba = demo_model.predict_proba(demo_Xte)[:, 1]
print("with-demographics test ROC-AUC:", round(roc_auc_score(y_test, demo_proba), 4))
demo_rates = {}
for s, label in [(1, 'Male'), (2, 'Female')]:
    mask = (sex_test.values == s)
    demo_rates[label] = (demo_proba[mask] < best_t).mean()
    print(f"{label}: approval_rate={demo_rates[label]:.4f}")
print("with-demographics adverse impact ratio:", round(min(demo_rates.values()) / max(demo_rates.values()), 4))


with-demographics test ROC-AUC: 0.7787
Male: approval_rate=0.2860
Female: approval_rate=0.3394
with-demographics adverse impact ratio: 0.8428


## 11. Hyperparameter tuning via cross-validation

In [14]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

param_dist = {
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'n_estimators': [100, 200, 300, 500],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
}
base_xgb = xgb.XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(base_xgb, param_distributions=param_dist, n_iter=30, scoring='roc_auc',
                             cv=cv, random_state=42, n_jobs=-1)
search.fit(X_train, y_train)
print("best params:", search.best_params_)
print("best CV ROC-AUC:", round(search.best_score_, 4))

tuned_model = search.best_estimator_
proba_tuned = tuned_model.predict_proba(X_test)[:, 1]
print("tuned test ROC-AUC:", round(roc_auc_score(y_test, proba_tuned), 4))
print("tuned test PR-AUC:", round(average_precision_score(y_test, proba_tuned), 4))
print("tuned test Brier:", round(brier_score_loss(y_test, proba_tuned), 4))
print("untuned test ROC-AUC (for comparison):", round(roc_auc_score(y_test, proba_xgb), 4))

best params: {'subsample': 1.0, 'n_estimators': 100, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.6}
best CV ROC-AUC: 0.786
tuned test ROC-AUC: 0.7807
tuned test PR-AUC: 0.5539
tuned test Brier: 0.1349
untuned test ROC-AUC (for comparison): 0.778


## 12. Intersectional fairness check: sex crossed with age group

In [15]:
audit_df['age_group'] = np.where(age_test.values < 30, 'Under 30', '30 and over')
audit_df['limit'] = X_test['LIMIT_BAL'].values

intersect_results = {}
for s, slabel in [(1, 'Male'), (2, 'Female')]:
    for ag in ['Under 30', '30 and over']:
        sub = audit_df[(audit_df['sex'] == s) & (audit_df['age_group'] == ag)]
        ar = (sub['pred_default'] == 0).mean()
        defaulters = sub[sub['actual'] == 1]
        fnr = defaulters['pred_default'].eq(0).mean() if len(defaulters) > 0 else float('nan')
        intersect_results[(slabel, ag)] = ar
        print(f"{slabel}, {ag}: n={len(sub)} approval_rate={ar:.4f} FNR={fnr:.4f} median_LIMIT_BAL={sub['limit'].median():,.0f}")

air_intersect = min(intersect_results.values()) / max(intersect_results.values())
print("\nintersectional adverse impact ratio:", round(air_intersect, 4))

Male, Under 30: n=669 approval_rate=0.2392 FNR=0.0467 median_LIMIT_BAL=80,000
Male, 30 and over: n=1733 approval_rate=0.3353 FNR=0.1046 median_LIMIT_BAL=150,000
Female, Under 30: n=1243 approval_rate=0.3001 FNR=0.1011 median_LIMIT_BAL=100,000
Female, 30 and over: n=2355 approval_rate=0.3325 FNR=0.1043 median_LIMIT_BAL=170,000

intersectional adverse impact ratio: 0.7134
